In [ ]:
!pip install langchain_community
# !pip install unstructured[docx]
!pip install accelerate
!pip install langchain
# !pip install pypdf
!pip install -U bitsandbytes
# !pip install -U git+https://github.com/huggingface/peft.git
# !pip install -U git+https://github.com/huggingface/accelerate.git
!pip install -U einops
!pip install -U safetensors
!pip install -U xformers
!pip install -U ctransformers[cuda]
# !pip install huggingface_hub
!pip install chromadb
!pip install sentence-transformers

In [ ]:
CHROMA_PATH = "your db path"
DATA_PATH = "data which you want to save"
EMBEDDING_PATH = '/content/drive/MyDrive/juris/multilingual-e5-large' #Embedding which you want to use
CHUNK_SIZE = 750
CHUNK_OVERLAP = 200

In [ ]:
from langchain.document_loaders import DirectoryLoader, TextLoader, CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, RecursiveJsonSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.schema import Document
from langchain.document_loaders.json_loader import JSONLoader

In [ ]:
import re
# def get_embeddings():
#    embeddings_hf = HuggingFaceEmbeddings(
#        model_name=EMBEDDING_PATH,
#        model_kwargs={"device": "cpu"},
#    )
#    return embeddings_hf
def load_documents():
  print('eheh')
  loader = DirectoryLoader(DATA_PATH, glob="*.docx")
  documents = loader.load()
  print(documents)
  return documents
def get_embeddings():
    embeddings_hf = HuggingFaceEmbeddings(
        model_name=EMBEDDING_PATH,
        model_kwargs={"device": "cuda"},
    )
    return embeddings_hf

# def load_documents():
#     # loader = DirectoryLoader('/content/drive/MyDrive/fix', glob='**/*.json', show_progress=True, loader_cls=JSONLoader, loader_kwargs = {'jq_schema':''})
#     loader = CSVLoader("/content/drive/MyDrive/fix/Сервисы EGOV.csv")
#     documents = loader.load()

#     return documents

def save_to_chroma(chunks):
    db = Chroma.from_documents(
        chunks, get_embeddings(), persist_directory=CHROMA_PATH
    )
    db.persist()
    print(f"Saved {len(chunks)} chunks to {CHROMA_PATH}.")

def split_text(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        add_start_index=True,
    )
    json_splitter = RecursiveJsonSplitter(max_chunk_size=500)
    chunks = []
    for doc in documents:
        # print(doc)
        # Split the document into chunks
        file_path = doc.metadata.get("source")
        last_four_characters = file_path[-4:]

        # Check if the last 4 characters are '.json'
        if last_four_characters.lower() == '.json':
          doc_chunks = json_splitter.split_json(doc.page_content)
        else:
          doc_chunks = text_splitter.split_text(doc.page_content)
        file_name = re.search(r'/([^/]+)\.(docx|csv|json|pdf)$', file_path).group(1)
        # Add metadata to each chunk
        for i, chunk in enumerate(doc_chunks):
            print(chunk)
            # Extracting the file name without extension

            print(file_name)
            chunk_metadata = {
                "file_name": file_name,  # Assuming the source key contains the file name
                "chunk_index": i,
            }
            chunks.append(Document(page_content=chunk, metadata=chunk_metadata))
            # print(chunk_metadata)

    print(f"Split {len(documents)} documents into {len(chunks)} chunks.")
    return chunks


In [ ]:
documents = load_documents()

In [ ]:
chunks = split_text(documents)

In [ ]:
save_to_chroma(chunks)